# Empirical Mixed-Type Landscape

`data_types` accepts a different type per variable, so a single landscape can mix `categorical`, `ordinal`, and `boolean` axes.

As a demo, we use the benzaldehyde electroreduction screen from:

Jacob B Eisenberg *et al.*, "Understanding the competition between alcohol formation and dimerization during electrochemical reduction of aromatic carbonyl compounds," *J. Am. Chem. Soc.* (2025). A 4 × 3 × 3 = 36-run factorial of electrode material × pH × applied potential.

`electrode` is nominal (`categorical`); `pH` and `potential_V` are numeric grids (`ordinal`).

## 1. Load the data

In [6]:
import pandas as pd

df = pd.read_csv('../data/Chemistry/EChem/aldehyde_electro_36.csv')
df[['electrode', 'pH', 'potential_V', 'selectivity_scalar']].head()

,electrode,pH,potential_V,selectivity_scalar
0,Bi,2,-1.0,70.56
1,Bi,2,-0.8,76.28
2,Bi,2,-0.6,71.21
3,Cu,2,-1.0,-25.95
4,Cu,2,-0.8,-30.22


## 2. Build the landscape

In [2]:
from graphfla.landscape import Landscape

X = df[['electrode', 'pH', 'potential_V']]
f = df['selectivity_scalar']

landscape = Landscape(maximize=True)
landscape.build_from_data(
    X, f,
    data_types={
        'electrode':   'categorical',
        'pH':          'ordinal',
        'potential_V': 'ordinal',
    },
    verbose=True,
)

Building Landscape from data...
 - Validating data types dictionary...
   - Data types dictionary validation successful.
 - Handling missing values and duplicates...
 - Checking for missing values...
 - Handling duplicate configurations...
   - No duplicate configurations found.
Preparing data for landscape construction (encoding variables)...
Constructing landscape graph...
# Constructing neighborhoods (active): 100%|██████████| 36/36 [00:00<00:00, 186413.51it/s]
 - Identified 102 improving connections.
 - Constructing graph object...
 - Adding node attributes (fitness, etc.)...
Calculating landscape properties...
 - Calculating network metrics (degrees)...
 - Determining local optima...
   - Found 3 local optima.
 - Determining global optimum...
   - Global optimum found at index 25 with fitness 100.0000.
Landscape built successfully.



Landscape(kind='default', maximize=True)

## 3. Basic information

In [3]:
print(f"Size: {len(landscape)}")
print(f"Dimension: {landscape.n_vars}")
print(f"Number of local optima: {landscape.n_lo}")
print(f"Global optimum index: {landscape.go_index}")
print("Global optimum information:")
print(landscape[landscape.go_index])

Size: 36
Dimension: 3
Number of local optima: 3
Global optimum index: 25
Global optimum information:
{'electrode': 'Bi', 'pH': np.int64(13), 'potential_V': np.float64(-0.8), 'fitness': np.float64(100.0), 'in_degree': 6, 'out_degree': 0, 'is_lo': True}


## 4. Landscape features

In [4]:
from graphfla.analysis import (
    local_optima_ratio, autocorrelation, fdc,
)

print(f"lo_ratio:        {local_optima_ratio(landscape):.4f}")
print(f"autocorrelation: {autocorrelation(landscape):.4f}")
print(f"fdc:             {fdc(landscape):.4f}")


 - Calculating distances to global optimum using mixed_distance...
   - Distances to GO calculated and added as node attribute 'dist_go'.


lo_ratio:        0.0833
autocorrelation: 0.3641
fdc:             -0.4057


### The whole feature panel in one call

`analysis.profile()` runs every whole-landscape metric and returns a tidy `pandas` Series — the computed-metric companion to `landscape.describe()`. Pass a list of landscapes to get a `DataFrame`, one row each, for side-by-side comparison.

In [ ]:
from graphfla import analysis

# one call -> a pandas Series of every whole-landscape metric
analysis.profile(landscape)

## 5. Per-axis mutation effects

In [5]:
from graphfla.analysis import single_mutation_effects

for axis in ['electrode', 'pH', 'potential_V']:
    print(f"\n=== {axis} ===")
    print(single_mutation_effects(landscape, position=axis))


=== electrode ===
  mutation_from mutation_to  median_abs_effect  mean_effect   p_value  \
0            Bi          Cu           0.965399    57.633333  0.001953   
1            Bi          Pb           0.708774    38.325556  0.019531   
2            Bi    graphite           0.441286     0.278889  0.253906   
3            Cu          Pb           0.708386   -19.307778  0.980469   
4            Cu    graphite           1.300583   -57.354444  1.000000   
5            Pb    graphite           0.980723   -38.046667  0.980469   

   significant  
0         True  
1         True  
2        False  
3        False  
4        False  
5        False  

=== pH ===
   mutation_from  mutation_to  median_abs_effect  mean_effect   p_value  \
0              2            7           0.508789    35.685000  0.019287   
1              2           13           0.648061   -35.028333  0.999756   
2              7           13           1.016123   -70.713333  1.000000   

   significant  
0         True  
1  

In [13]:
from graphfla.analysis import walsh_hadamard

walsh_hadamard(landscape)

... Total theoretical features (order:count): 2:16
... Total retained features (order:count): 2:16 (100.0%)
Construction time for H_matrix...


,order,positions,term,coefficient
0,0,(),WT,33.079444
1,1,"(1,)",1_1_2,-57.633333
2,1,"(1,)",1_1_3,-38.325556
3,1,"(1,)",1_1_4,-0.278889
4,1,"(2,)",1_2_2,-35.685000
5,1,"(2,)",1_2_3,35.028333
6,1,"(3,)",1_3_2,24.071667
7,1,"(3,)",1_3_3,50.041667
8,2,"(1, 2)",1_1_2-1_2_2,60.853333
9,2,"(1, 2)",1_1_2-1_2_3,34.276667
